# BulkFluo RDF — four-color granule SACD

Segments shared RNA granules from the equal-weight 488/561/647 aggregate, rejects diffraction-limited single molecules, exports four-channel crops, and calculates size-normalized RDF curves through 1.3 equivalent radii.

## Parameters

In [ ]:
from pathlib import Path
import copy, os, yaml

DATASET = Path('/Volumes/guttman/users/gmgao/Imaging_ProcessedData/Collaborations/QingTang/20260811_ONI-gmgao-forQing_CB_4color')
OUTPUT = DATASET / 'BulkFluoRDF_granuleSACD_results'
params = {
    'pixel_size_nm': 58.5,
    'crop_padding_px': 10,
    'background_sigma_px': 12.0,
    'smooth_sigma_px': 1.5,
    'normalization_upper_percentile': 99.7,
    'watershed_min_distance_px': 5,
    'minimum_support_channels': 2,
    'minimum_channel_support_fraction': 0.10,
    'minimum_equivalent_diameter_px': 15.0,
    'rdf_maximum_normalized_radius': 1.3,
    'rdf_bin_width': 0.10,
    'rdf_bin_step': 0.05,
    'rdf_subpixel_samples_per_axis': 8,
}
params

## Build and save the reproducible configuration

In [ ]:
import importlib
import BulkFluoRDF_granuleSACD as granule
granule = importlib.reload(granule)

cfg = copy.deepcopy(granule.DEFAULT_CONFIG)
cfg.update({'input_dir': str(DATASET), 'output_dir': str(OUTPUT), 'expected_fovs': 10,
            'pixel_size_nm': params['pixel_size_nm'], 'crop_padding_px': params['crop_padding_px']})
cfg['segmentation'].update({key: params[key] for key in [
    'background_sigma_px', 'smooth_sigma_px', 'normalization_upper_percentile',
    'watershed_min_distance_px', 'minimum_support_channels',
    'minimum_channel_support_fraction', 'minimum_equivalent_diameter_px']})
cfg['rdf'].update({'maximum_normalized_radius': params['rdf_maximum_normalized_radius'],
                   'bin_width': params['rdf_bin_width'], 'bin_step': params['rdf_bin_step'],
                   'subpixel_samples_per_axis': params['rdf_subpixel_samples_per_axis']})
OUTPUT.mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(OUTPUT / '.cache' / 'matplotlib')
RUN_CONFIG = OUTPUT / 'run_config.generated.yaml'
RUN_CONFIG.write_text(yaml.safe_dump(cfg, sort_keys=False))
RUN_CONFIG

## Input validation and synthetic acceptance checks

In [ ]:
granule.synthetic_validation()
pairs = granule.pair_four_channel_files(DATASET)
assert len(pairs) == 10
print(f'{len(pairs)} complete FOVs / {len(pairs) * 4} input TIFFs')
[{p.fov: {ch: p.paths[ch].name for ch in granule.CHANNELS}} for p in pairs][0]

## Run the complete dataset

In [ ]:
result = granule.run_pipeline(RUN_CONFIG)
print(f'Processed {len(result.pairs)} FOVs')
print(f'Retained {int(result.properties.keep_granule.sum())} granules >15 px')
result.properties.head()

## Segmentation QC and retained/rejected size distributions

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUTPUT / 'granule_size_distribution.png')))
for path in sorted((OUTPUT / 'qc_overlays').glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))

## Four-channel crop preview

In [ ]:
import matplotlib.pyplot as plt
import tifffile
crop_path = sorted((OUTPUT / 'granule_crops').glob('*.tif'))[0]
crop = tifffile.imread(crop_path)
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, image, channel in zip(axes, crop, granule.CHANNELS):
    ax.imshow(image, cmap='gray', vmax=float(__import__('numpy').percentile(image, 99.5)))
    ax.set_title(channel); ax.axis('off')
fig.suptitle(crop_path.name); plt.show()

## RDF tables, aggregate curves, and production summary

In [ ]:
import json
display(result.rdf.head())
display(result.aggregate.head())
display(result.correlations.head())
display(Image(filename=str(OUTPUT / 'rdf_aggregate.png')))
display(Image(filename=str(OUTPUT / 'granule_rdf_correlation_distributions.png')))
display(Image(filename=str(OUTPUT / 'granule_rdf_correlation_violin_box_statannotations.png')))
display(Image(filename=str(OUTPUT / 'granule_rdf_correlation_violin_box_statannotations_paired_ttest.png')))
for path in sorted((OUTPUT / 'correlation_representatives').glob('*/*__rdf.png')):
    print(path.parent.name)
    display(Image(filename=str(path)))
json.loads((OUTPUT / 'full_dataset_run_summary.json').read_text())